<a href="https://colab.research.google.com/github/chawlaniharika23/Fraud_Detection_Research/blob/main/notebooks/05_model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import load_npz
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

base_dir = "/content/drive/MyDrive/fraud-detection-research"
processed_dir = f"{base_dir}/data/processed"
recovery_dir = f"{processed_dir}/recovered"
base_model_dir = f"{base_dir}/results/base_models"
stacking_dir = f"{base_dir}/results/dynamic_stacking"

os.makedirs(stacking_dir, exist_ok=True)

X_val = load_npz(
    f"{recovery_dir}/ieee_X_validation_selected_100.npz"
)

y_val = pd.read_parquet(
    f"{processed_dir}/ieee_y_validation.parquet"
).iloc[:, 0].to_numpy()

model_names = ["LR", "DT", "RF", "GBT", "NN"]

models = {
    name: joblib.load(f"{base_model_dir}/{name}_model.joblib")
    for name in model_names
}

print("Validation shape:", X_val.shape)
print("Labels shape:", y_val.shape)
print("Loaded models:", list(models))

Mounted at /content/drive
Validation shape: (88581, 100)
Labels shape: (88581,)
Loaded models: ['LR', 'DT', 'RF', 'GBT', 'NN']


In [ ]:
#Generate base-model predictions
prediction_columns = {}

for name, model in models.items():
    prediction_columns[name] = model.predict_proba(X_val)[:, 1]
    print(f"{name}: predictions generated")

prediction_df = pd.DataFrame(prediction_columns)

print("\nPrediction matrix:", prediction_df.shape)
display(prediction_df.head())

LR: predictions generated
DT: predictions generated
RF: predictions generated
GBT: predictions generated
NN: predictions generated

Prediction matrix: (88581, 5)


,LR,DT,RF,GBT,NN
0,0.097680,0.167711,0.231922,0.017780,0.006320
1,0.223986,0.228284,0.205386,0.016057,0.004117
2,0.274524,0.035514,0.301206,0.017430,0.013096
3,0.106156,0.099543,0.083251,0.011851,0.001527
4,0.581341,0.378281,0.231324,0.025982,0.022744


In [ ]:
#Divide validation data into two parts
indices = np.arange(len(y_val))

weight_idx, meta_idx = train_test_split(
    indices,
    test_size=0.5,
    stratify=y_val,
    random_state=42
)

P_weight = prediction_df.iloc[weight_idx].to_numpy()
P_meta = prediction_df.iloc[meta_idx].to_numpy()

y_weight = y_val[weight_idx]
y_meta = y_val[meta_idx]

print("Weight-estimation rows:", len(weight_idx))
print("Meta-training rows:", len(meta_idx))
print("Weight-estimation fraud cases:", int(y_weight.sum()))
print("Meta-training fraud cases:", int(y_meta.sum()))

Weight-estimation rows: 44290
Meta-training rows: 44291
Weight-estimation fraud cases: 1550
Meta-training fraud cases: 1550


In [ ]:
#Calculate dynamic weights
def feature_importance_vector(model):
    if hasattr(model, "feature_importances_"):
        importance = np.abs(model.feature_importances_)

    elif hasattr(model, "coef_"):
        importance = np.abs(model.coef_).mean(axis=0)

    elif hasattr(model, "coefs_"):
        importance = np.abs(model.coefs_[0]).sum(axis=1)

    else:
        raise ValueError(
            f"No supported importance method: {type(model).__name__}"
        )

    importance = np.asarray(importance).ravel()
    total = importance.sum()

    if total <= 0:
        return np.ones_like(importance) / len(importance)

    return importance / total


weight_records = []

for j, name in enumerate(model_names):
    importance = feature_importance_vector(models[name])

    top20_concentration = np.sort(importance)[-20:].sum()

    pr_auc = average_precision_score(
        y_weight,
        P_weight[:, j]
    )

    raw_weight = pr_auc * top20_concentration

    weight_records.append({
        "Model": name,
        "Weight_Set_PR_AUC": pr_auc,
        "Top20_Importance_Concentration": top20_concentration,
        "Raw_Weight": raw_weight
    })

weights_df = pd.DataFrame(weight_records)

weights_df["Dynamic_Weight"] = (
    weights_df["Raw_Weight"]
    / weights_df["Raw_Weight"].sum()
)

weights = weights_df["Dynamic_Weight"].to_numpy()

weights_df.to_csv(
    f"{stacking_dir}/dynamic_weights.csv",
    index=False
)

display(weights_df.round(4))
print("Weight sum:", round(weights.sum(), 6))

,Model,Weight_Set_PR_AUC,Top20_Importance_Concentration,Raw_Weight,Dynamic_Weight
0,LR,0.3529,0.6928,0.2445,0.1680
1,DT,0.4389,0.8435,0.3702,0.2544
2,RF,0.5454,0.4994,0.2724,0.1871
3,GBT,0.4163,0.9548,0.3975,0.2731
4,NN,0.5514,0.3100,0.1709,0.1174


Weight sum: 1.0


In [ ]:
#Train the meta-learner
P_meta_weighted = P_meta * weights

meta_model = LogisticRegression(
    class_weight="balanced",
    max_iter=500,
    random_state=42
)

meta_model.fit(P_meta_weighted, y_meta)

joblib.dump(
    meta_model,
    f"{stacking_dir}/meta_model.joblib"
)

joblib.dump(
    {
        "model_names": model_names,
        "weights": weights,
        "method": "validation-holdout feature-importance-aware stacking",
        "random_state": 42
    },
    f"{stacking_dir}/stacking_config.joblib"
)

print("Meta-learner trained and saved.")
print("Meta-training rows:", P_meta_weighted.shape[0])
print("Meta-features:", P_meta_weighted.shape[1])

Meta-learner trained and saved.
Meta-training rows: 44291
Meta-features: 5


In [ ]:
#Reload and verify stacking components

import os
import joblib
import pandas as pd
import numpy as np

base_dir = "/content/drive/MyDrive/fraud-detection-research"
stacking_dir = f"{base_dir}/results/dynamic_stacking"
base_model_dir = f"{base_dir}/results/base_models"

weights_df = pd.read_csv(
    f"{stacking_dir}/dynamic_weights.csv"
)

config = joblib.load(
    f"{stacking_dir}/stacking_config.joblib"
)

meta_model = joblib.load(
    f"{stacking_dir}/meta_model.joblib"
)

display(weights_df.round(4))

print("Models:", config["model_names"])
print("Weights sum:", round(sum(config["weights"]), 6))
print("Meta-learner:", type(meta_model).__name__)

,Model,Weight_Set_PR_AUC,Top20_Importance_Concentration,Raw_Weight,Dynamic_Weight
0,LR,0.3529,0.6928,0.2445,0.1680
1,DT,0.4389,0.8435,0.3702,0.2544
2,RF,0.5454,0.4994,0.2724,0.1871
3,GBT,0.4163,0.9548,0.3975,0.2731
4,NN,0.5514,0.3100,0.1709,0.1174


Models: ['LR', 'DT', 'RF', 'GBT', 'NN']
Weights sum: 1.0
Meta-learner: LogisticRegression


In [ ]:
#Extract and save feature importance
# Load feature names using the correct column
selected_names = pd.read_csv(
    f"{base_dir}/data/processed/recovered/ieee_top_100_features.csv"
)["feature_name"].tolist()

print("Selected feature count:", len(selected_names))

models = {
    name: joblib.load(
        f"{base_model_dir}/{name}_model.joblib"
    )
    for name in config["model_names"]
}

importance_records = []

for name, model in models.items():

    if hasattr(model, "feature_importances_"):
        importance = np.abs(model.feature_importances_)

    elif hasattr(model, "coef_"):
        importance = np.abs(model.coef_).mean(axis=0)

    elif hasattr(model, "coefs_"):
        importance = np.abs(model.coefs_[0]).sum(axis=1)

    else:
        print(f"Skipping unsupported model: {name}")
        continue

    importance = importance / (importance.sum() + 1e-12)

    if len(importance) != len(selected_names):
        raise ValueError(
            f"{name}: importance count {len(importance)} "
            f"does not match feature count {len(selected_names)}"
        )

    for feature, score in zip(selected_names, importance):
        importance_records.append({
            "Model": name,
            "Feature": feature,
            "Importance": score
        })

importance_df = pd.DataFrame(importance_records)

importance_df.to_csv(
    f"{stacking_dir}/base_model_feature_importance.csv",
    index=False
)

print("\nFeature importance saved successfully.")
print("Total records:", len(importance_df))

display(
    importance_df.sort_values(
        "Importance", ascending=False
    ).groupby("Model").head(5)
)

Selected feature count: 100

Feature importance saved successfully.
Total records: 500


,Model,Feature,Importance
334,GBT,V201,0.276499
101,DT,V258,0.243323
320,GBT,C1,0.164922
103,DT,V70,0.137932
301,GBT,V258,0.115957
2,LR,C14,0.102305
24,LR,C11,0.092645
102,DT,C14,0.087382
302,GBT,C14,0.082865
113,DT,V294,0.082661


In [ ]:
#Compare weighted and unweighted stacking
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score
)

# Recreate the original validation partition
X_val = load_npz(
    f"{base_dir}/data/processed/recovered/ieee_X_validation_selected_100.npz"
)

y_val = pd.read_parquet(
    f"{base_dir}/data/processed/ieee_y_validation.parquet"
).iloc[:, 0].to_numpy()

indices = np.arange(len(y_val))

weight_idx, meta_idx = train_test_split(
    indices,
    test_size=0.5,
    stratify=y_val,
    random_state=42
)

# Generate predictions
P_val = np.column_stack([
    models[name].predict_proba(X_val)[:, 1]
    for name in config["model_names"]
])

# Split meta-training data into fitting and comparison portions
meta_train_idx, meta_eval_idx = train_test_split(
    np.arange(len(meta_idx)),
    test_size=0.5,
    stratify=y_val[meta_idx],
    random_state=42
)

P_meta = P_val[meta_idx]
y_meta = y_val[meta_idx]

P_fit = P_meta[meta_train_idx]
y_fit = y_meta[meta_train_idx]

P_eval = P_meta[meta_eval_idx]
y_eval = y_meta[meta_eval_idx]

# Weighted meta-learner
weighted_model = LogisticRegression(
    class_weight="balanced",
    max_iter=500,
    random_state=42
)

weighted_model.fit(
    P_fit * np.asarray(config["weights"]),
    y_fit
)

weighted_prob = weighted_model.predict_proba(
    P_eval * np.asarray(config["weights"])
)[:, 1]

# Unweighted meta-learner
unweighted_model = LogisticRegression(
    class_weight="balanced",
    max_iter=500,
    random_state=42
)

unweighted_model.fit(P_fit, y_fit)

unweighted_prob = unweighted_model.predict_proba(
    P_eval
)[:, 1]

comparison = []

for name, probabilities in [
    ("Weighted stacking", weighted_prob),
    ("Unweighted stacking", unweighted_prob)
]:
    predictions = (probabilities >= 0.5).astype(int)

    comparison.append({
        "Method": name,
        "Precision": precision_score(
            y_eval, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_eval, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_eval, predictions, zero_division=0
        ),
        "ROC_AUC": roc_auc_score(y_eval, probabilities),
        "PR_AUC": average_precision_score(y_eval, probabilities)
    })

comparison_df = pd.DataFrame(comparison)

comparison_df.to_csv(
    f"{stacking_dir}/weighted_vs_unweighted.csv",
    index=False
)

display(comparison_df.round(4))

print("Comparison saved successfully.")

,Method,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Weighted stacking,0.1796,0.7652,0.2909,0.9038,0.5793
1,Unweighted stacking,0.1949,0.7639,0.3106,0.9104,0.5943


Comparison saved successfully.


In [ ]:
#Save the experiment summary
import json
import os
import numpy as np

summary = {
    "experiment": "Dynamic Stacking for IEEE-CIS Fraud Detection",
    "method": "Feature-importance-aware validation-holdout stacking",
    "base_models": config["model_names"],
    "number_of_base_models": 5,
    "selected_features": 100,
    "meta_learner": "Logistic Regression",
    "weight_estimation_rows": len(weight_idx),
    "meta_training_rows": len(meta_idx),
    "dynamic_weights": {
        name: float(weight)
        for name, weight in zip(
            config["model_names"],
            config["weights"]
        )
    },
    "test_metrics": {
        "precision": 0.1916,
        "recall": 0.7944,
        "f1": 0.3088,
        "roc_auc": 0.9087,
        "pr_auc": 0.5839
    },
    "weighted_vs_unweighted_validation": {
        "weighted_pr_auc": float(
            comparison_df.loc[
                comparison_df["Method"] == "Weighted stacking",
                "PR_AUC"
            ].iloc[0]
        ),
        "unweighted_pr_auc": float(
            comparison_df.loc[
                comparison_df["Method"] == "Unweighted stacking",
                "PR_AUC"
            ].iloc[0]
        )
    },
    "limitations": [
        "Five base learners used instead of six in the paper.",
        "Validation-holdout stacking used instead of full out-of-fold stacking.",
        "SVM excluded from primary stacking because its calibration used validation labels.",
        "Feature-importance weighting is an implementation assumption.",
        "Unweighted stacking performed better in the exploratory validation comparison."
    ]
}

with open(
    f"{stacking_dir}/stacking_experiment_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=4)

print("Notebook 5 summary saved.")
print("Files in dynamic stacking folder:")

for filename in sorted(os.listdir(stacking_dir)):
    print("-", filename)

Notebook 5 summary saved.
Files in dynamic stacking folder:
- base_model_feature_importance.csv
- dynamic_weights.csv
- meta_model.joblib
- stacking_config.joblib
- stacking_experiment_summary.json
- weighted_vs_unweighted.csv
